<a href="https://colab.research.google.com/github/kunphat510214-netizen/Final-Final/blob/step-7-8/Coffee_Shop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ขั้นตอนที่ 1 เข้าใจองค์กรของกลุ่ม
1. องค์กรนี้มีสิ่งของหลักอะไรบ้างที่ต้องเก็บข้อมูล?
ร้านกาแฟมี ”ออเดอร์เครื่องดื่ม“ เป็นสิ่งหลัก
2. แต่ละสิ่งของหลักนั้น มี "คุณสมบัติ" (ข้อมูล) อะไรบ้าง?
ออเดอร์มี : รหัสออเดอร์, ชื่อลูกค้า,ชื่อเมนู/ขนาด/ท็อปปิ้ง/ระดับความหวาน
3. แต่ละสิ่งของหลักนั้น ทำอะไรได้บ้าง?
ออเดอร์ “คำนวณราคา”ได้,ชื่อเมนู/ขนาด/ท็อปปิ้ง “เช็กว่าวัตถุดิบของเมนูหรือขนาดนั้นๆ หมดหรือยัง”ได้
4. ธุรกิจนี้มีการตัดสินใจหรือคำนวณอะไรที่ซับซ้อนกว่าการบวกเลขตรงตรงบ้าง?
การคำนวณที่ซับซ้อน:ราคาตามขนาดและท็อปปิ้ง (ขนาดใหญ่ +10 บาท/แก้ว, เปลี่ยนเป็นนมอัลมอนด์ +15 บาท/แก้ว)
 การคิดส่วนลดตามโปรโมชันหรือสมาชิก (เช่น สมาชิกสะสมแต้มครบ 10 แก้ว ฟรี 1 แก้ว)

# ขั้นตอนที่ 2 -- ออกแบบ Class

*   List item
*   List item


ออกแบบ Class — สร้างอย่างน้อย 3 class ที่จำลอง "สิ่งของ" หลักในธุรกิจ แต่ละ class ต้องมี attribute และ method ที่สมเหตุสมผล

In [1]:
# ---------------------------------------
# 1. การนำเข้าไลบรารี (Import Libraries)
# ---------------------------------------
import random
import sqlite3
import pandas as pd

# ตรวจสอบการนำเข้า matplotlib สำหรับวาดกราฟ (ป้องกันโปรแกรมค้างหากไม่ได้ติดตั้ง)
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

#  ตรวจสอบการนำเข้า display สำหรับแสดงผลใน Jupyter Notebook/Colab
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

# --------------------------------------
# 2. คลาสสำหรับจัดการข้อมูลสมาชิก (Member)
# --------------------------------------
class Member:
    def __init__(self, member_id, name, points=0):
        #  กำหนดค่าเริ่มต้นให้กับสมาชิก: รหัสสมาชิก, ชื่อ, และแต้มสะสมเริ่มต้น
        self.member_id = member_id
        self.name = name
        self.points = points #  แต้มสะสมแรกรับ

    def add_points(self, cups=1):
        # ฟังก์ชันสำหรับสะสมแต้มตามจำนวนแก้วที่สั่งซื้อ
        self.points += cups
        return self.points #  คืนค่าแต้มสะสมล่าสุด

# ----------------------------------------------
# 3.  คลาสสำหรับจัดการออเดอร์เครื่องดื่ม (DrinkOrder)
# ----------------------------------------------
class DrinkOrder:
    # --- กำหนดราคามาตรฐานและตัวเลือกต่างๆ (Class Attributes) ---
    BASE_PRICES = {
        "อเมริกาโน่": 45, "ลาเต้": 50, "คาปูชิโน่": 50,
        "เอสเพรสโซ่": 50, "มอคค่า": 50, "ชาเขียว": 45,
        "ชาไทย": 45, "มัทฉะลาเต้": 50, "ช็อกโกแลต": 50,
        "นมสด": 40, "สตรอว์เบอร์รี่โซดา": 40,
        "บลูฮาวายโซดา": 40, "ยูซุโซดา": 40,
    }
    TYPE_PRICES = {"ร้อน": 0, "เย็น": 5, "ปั่น": 10}
    SIZE_PRICES = {"S": 0, "M": 10, "L": 20}
    TOPPING_PRICES = {"ไม่มี": 0, "ไข่มุก": 5, "วิปครีม": 15, "ครีมชีส": 20, "เจลลี่": 5}
    PAYMENT_METHODS = ("เงินสด", "QR Code", "บัตร")

    def __init__(self, order_id, customer_name, menu_name, drink_type, size,
                 topping, sweetness_level, receive_type, member_id=None):
        #  จัดเก็บข้อมูลและรายละเอียดออเดอร์
        self.order_id = order_id
        self.queue_no = f"Q{order_id:03d}"   #  สร้างหมายเลขคิว เช่น Q001
        self.customer_name = customer_name
        self.menu_name = menu_name
        self.drink_type = drink_type
        self.size = size
        self.topping = topping
        self.sweetness_level = sweetness_level
        self.receive_type = receive_type
        self.member_id = member_id

        #  สถานะเริ่มต้นของการชำระเงินและการดำเนินงาน
        self.payment_method = None
        self.payment_status = "ยังไม่ชำระ"
        self.status = "รอชำระเงิน"
        self.wait_minutes = 0

    def calculate_subtotal(self):
        #  คำนวณราคารวมขั้นต้น (ราคาฐาน + ประเภทเครื่องดื่ม + ขนาด + ท็อปปิ้ง)
        return (self.BASE_PRICES[self.menu_name]
                + self.TYPE_PRICES[self.drink_type]
                + self.SIZE_PRICES[self.size]
                + self.TOPPING_PRICES[self.topping])

    def calculate_price(self):
        # คำนวณราคาสุทธิ (หากเป็นสมาชิกจะได้รับส่วนลด 5%)
        # สมาชิกได้รับส่วนลด 5%
        discount_rate = 0.05 if self.member_id else 0
        return round(self.calculate_subtotal() * (1 - discount_rate), 2)

    def confirm_payment(self, payment_method):
        # ยืนยันการชำระเงินและอัปเดตสถานะออเดอร์
        if payment_method not in self.PAYMENT_METHODS:
            raise ValueError("ช่องทางชำระเงินไม่ถูกต้อง")
        self.payment_method = payment_method
        self.payment_status = "ชำระแล้ว"
        self.status = "รอดำเนินการ"

    def receipt_text(self):
        #  ข้อความรูปแบบใบเสร็จรับเงิน
        return (f"ใบเสร็จ R{self.order_id:05d} | {self.queue_no} | "
                f"{self.menu_name} {self.drink_type} {self.size} | "
                f"{self.calculate_price():,.2f} บาท | {self.payment_method}")

    def sticker_text(self):
        # ข้อความรูปแบบสติกเกอร์สำหรับติดแก้วน้ำ
        return (f"{self.queue_no} {self.menu_name}/{self.drink_type}/{self.size} "
                f"หวาน {self.sweetness_level} ท็อปปิ้ง {self.topping} ({self.receive_type})")

    def to_record(self):
        # แปลงข้อมูลออเดอร์เป็น Dictionary เพื่อสะดวกในการบันทึกลงฐานข้อมูลหรือนำไปสร้าง DataFrame
        return {
            "order_id": self.order_id,
            "queue_no": self.queue_no,
            "customer_name": self.customer_name,
            "member_id": self.member_id,
            "menu_name": self.menu_name,
            "drink_type": self.drink_type,
            "size": self.size,
            "topping": self.topping,
            "sweetness_level": self.sweetness_level,
            "receive_type": self.receive_type,
            "payment_method": self.payment_method,
            "subtotal": self.calculate_subtotal(),
            "price": self.calculate_price(),
            "wait_minutes": self.wait_minutes,
            "status": self.status,
        }

# ----------------------------------------------
# 4. คลาสสำหรับจัดการระบบคิวและครัว (QueueSystem)
# ----------------------------------------------
class QueueSystem:
    #  รายการออเดอร์ที่กำลังรอดำเนินการ และประวัติเหตุการณ์ (Log)
    def __init__(self):
        self.active_orders = []
        self.events = []
    #  ฟังก์ชันภายในสำหรับบันทึกประวัติการเปลี่ยนแปลงสถานะของออเดอร์
    def _log(self, order, event):
        self.events.append({
            "event_id": len(self.events) + 1,
            "order_id": order.order_id,
            "event": event,
            "status": order.status,
        })

    def send_to_kds(self, order):
        #  ส่งออเดอร์เข้าระบบห้องครัว (Kitchen Display System) หลังจากชำระเงินแล้ว
        if order.payment_status != "ชำระแล้ว":
            raise ValueError("ต้องยืนยันการชำระเงินก่อนส่งเข้า KDS")
        self.active_orders.append(order)
        order.status = "รอดำเนินการ"
        self._log(order, "ส่งเข้า KDS")

    def start_preparing(self, order):
        #  บาริสต้าเริ่มชง/ทำตามออเดอร์
        order.status = "กำลังทำ"
        self._log(order, "บาริสต้ารับออเดอร์")

    def call_queue(self, order, announce=False):
        #  เรียกคิวเมื่อทำเครื่องดื่มเสร็จเรียบร้อย
        order.status = "พร้อมรับ"
        self._log(order, "เรียกคิว")
        message = f" คิว {order.queue_no} พร้อมรับที่เคาน์เตอร์"
        if announce:
            print(message, "(เสียงเรียก)")
        return message

    def complete_order(self, order):
        #  ส่งมอบเครื่องดื่มให้ลูกค้าเสร็จสิ้น และลบออกจากรายการคิวที่รอดำเนินการ
        order.status = "เสร็จสิ้น"
        self._log(order, "ส่งมอบสำเร็จ")
        self.active_orders = [o for o in self.active_orders if o.order_id != order.order_id]

## ขั้นที่ 3: เขียน functions ช่วยทำงาน


In [2]:
# ขั้นที่ 3: เขียน functions ช่วยทำงาน
import random

def generate_customer_name():
    first_names = ["เจสซี่", "ลีโอ", "มายา", "อเล็กซ์", "คลาร่า", "ลูคัส", "นิโคล"]
    last_names = ["รักเรียน", "ใจดี", "สายทอง", "รุ่งเรือง", "มั่นคง", "วงศ์สว่าง", "เจริญพร"]
    return f"{random.choice(first_names)} {random.choice(last_names)}"


In [3]:
# ฟังก์ชันเลือกค่าระดับความหวานแบบสุ่ม
def random_sweetness_level(levels=("0%", "25%", "50%", "75%", "100%")):
    # เลือกค่าระดับความหวานแบบสุ่มจากรายการที่กำหนดใน 'levels'
    # โดย 'levels' มีค่าเริ่มต้นเป็น ('0%', '25%', '50%', '75%', '100%') หากไม่ได้ระบุ
    return random.choice(levels)

In [4]:
# ฟังก์ชันสำหรับตรวจสอบสถานะคำสั่งซื้อ
def check_order_status(order):
    return order.status

In [5]:
# ฟังก์ชันสำหรับสร้างคำสั่งซื้อแบบสุ่ม
def build_random_order(order_id, member=None):
    return DrinkOrder(
        order_id=order_id,
        customer_name=member.name if member else generate_customer_name(),
        menu_name=random.choice(list(DrinkOrder.BASE_PRICES)),
        drink_type=random.choice(list(DrinkOrder.TYPE_PRICES)),
        size=random.choice(list(DrinkOrder.SIZE_PRICES)),
        topping=random.choice(list(DrinkOrder.TOPPING_PRICES)),
        sweetness_level=random_sweetness_level(),
        receive_type=random.choice(["ทานที่ร้าน", "กลับบ้าน"]),
        member_id=member.member_id if member else None,
    )

# ส่วนสาธิตการทำงานของระบบ (Demo)
# 1. สร้างสมาชิกตัวอย่าง
demo_member = Member("M001", "มายา ใจดี", points=4)
# 2. สร้างคำสั่งซื้อตัวอย่างสำหรับสมาชิก
demo_order = DrinkOrder(1, demo_member.name, "ลาเต้", "เย็น", "M", "วิปครีม",
                        "50%", "กลับบ้าน", demo_member.member_id)
# 3. สร้างระบบคิว
demo_queue = QueueSystem()

# แสดงข้อมูลการรับลูกค้าและตัวเลือกของเครื่องดื่ม
print("1-3) รับลูกค้าและตัวเลือก:", demo_order.menu_name, demo_order.size,
      demo_order.topping, demo_order.sweetness_level, demo_order.receive_type,
      "สมาชิก" if demo_order.member_id else "ไม่เป็นสมาชิก")
# 4. แสดงสรุปราคาเครื่องดื่ม
print("4) สรุปราคา:", demo_order.calculate_price(), "บาท")
# ยืนยันการชำระเงินด้วย QR Code
demo_order.confirm_payment("QR Code")
# สมาชิกได้รับคะแนนเพิ่ม
demo_member.add_points()
# 5. แสดงสถานะการชำระเงินและช่องทางการชำระเงิน
print("5) ชำระเงิน:", demo_order.payment_status, "ด้วย", demo_order.payment_method)
# 6. แสดงใบเสร็จ
print("6)", demo_order.receipt_text())
# แสดงข้อความบนสติกเกอร์
print("   Sticker:", demo_order.sticker_text())
# 7. ส่งคำสั่งซื้อเข้า KDS (Kitchen Display System)
demo_queue.send_to_kds(demo_order)
print("7) KDS:", demo_order.status)
# บาริสต้าเริ่มเตรียมเครื่องดื่ม
demo_queue.start_preparing(demo_order)
# เรียกคิวเมื่อเครื่องดื่มพร้อมรับ
demo_queue.call_queue(demo_order, announce=True)
# 8-9. แสดงหน้าจอคิวและสถานะคำสั่งซื้อ
print("8-9) หน้าจอแสดง:", demo_order.queue_no, demo_order.status)
# ทำเครื่องดื่มเสร็จสมบูรณ์
demo_queue.complete_order(demo_order)
# 10. แสดงสถานะสุดท้ายของคำสั่งซื้อและจำนวนคิวที่ยังคงเหลืออยู่
print("10) สถานะ:", demo_order.status, "| คิวคงเหลือ:", len(demo_queue.active_orders))

1-3) รับลูกค้าและตัวเลือก: ลาเต้ M วิปครีม 50% กลับบ้าน สมาชิก
4) สรุปราคา: 76.0 บาท
5) ชำระเงิน: ชำระแล้ว ด้วย QR Code
6) ใบเสร็จ R00001 | Q001 | ลาเต้ เย็น M | 76.00 บาท | QR Code
   Sticker: Q001 ลาเต้/เย็น/M หวาน 50% ท็อปปิ้ง วิปครีม (กลับบ้าน)
7) KDS: รอดำเนินการ
 คิว Q001 พร้อมรับที่เคาน์เตอร์ (เสียงเรียก)
8-9) หน้าจอแสดง: Q001 พร้อมรับ
10) สถานะ: เสร็จสิ้น | คิวคงเหลือ: 0


#ขั้นที่ 4 Loop

In [6]:
# ขั้นที่ 4: จำลองธุรกรรมทีละรายการด้วย loop อย่างน้อย 300 รายการ
random.seed(612104)
members = {
    f"M{i:03d}": Member(f"M{i:03d}", generate_customer_name(), random.randint(0, 9))
    for i in range(1, 61)
}

#สร้างระบบจัดการคิว และสร้างลิสต์สำหรับเก็บข้อมูลออเดอร์ทั้งหมด
queue_system = QueueSystem()
orders = []

#วนลูปเพื่อจำลองธุรกรรมจำนวน 300 รายการ
for i in range(1, 301):

  #สุ่มเลือกสมาชิก โดยมีโอกาส 40% ที่ออเดอร์จะเป็นของสมาชิก
    member = random.choice(list(members.values())) if random.random() < 0.40 else None

    #สร้างออเดอร์แบบสุ่ม พร้อมระบุหมายเลขออเดอร์และสมาชิก
    order = build_random_order(i, member)

    #ยืนยันการชำระเงินด้วยวิธีการชำระเงินแบบสุ่ม
    order.confirm_payment(random.choice(DrinkOrder.PAYMENT_METHODS))

    #หากเป็นสมาชิก ให้เพิ่มคะแนนสะสม
    if member:
        member.add_points()

   #ส่งออเดอร์เข้าสู่ระบบครัว (KDS)
    queue_system.send_to_kds(order)

    #เริ่มกระบวนการเตรียมเครื่องดื่ม
    queue_system.start_preparing(order)

    #สุ่มเวลารอรับเครื่องดื่มระหว่าง 2-18 นาที
    order.wait_minutes = random.randint(2, 18)

    #เรียกลูกค้าตามคิว
    queue_system.call_queue(order)

    #เปลี่ยนสถานะออเดอร์เป็นเสร็จสมบูรณ์
    queue_system.complete_order(order)

    #เก็บออเดอร์ที่ดำเนินการสร้างแล้วไว้ในลิสต์
    orders.append(order)

#ตรวจสอบว่ามีออเดอร์ครบ 300 รายการ
assert len(orders) == 300

#ตรวจสอบว่าไม่มีออเดอร์ค้างอยู่ในระบบ
assert len(queue_system.active_orders) == 0

#แสดงผลการจำลองธุรกรรม
print(f"จำลองครบ {len(orders)} ออเดอร์ และส่งมอบทุกคิวเรียบร้อย")

จำลองครบ 300 ออเดอร์ และส่งมอบทุกคิวเรียบร้อย


# ขั้นที่ 5-6: บันทึก CSV และฐานข้อมูล SQLite

In [7]:
# ขั้นที่ 5-6: บันทึก CSV และฐานข้อมูล SQLite 3 ตาราง
records = [order.to_record() for order in orders]
df = pd.DataFrame(records)
df.to_csv("coffee_orders.csv", index=False, encoding="utf-8-sig")

members_df = pd.DataFrame([
    {"member_id": member.member_id, "member_name": member.name, "points": member.points}
    for member in members.values()
])
events_df = pd.DataFrame(queue_system.events)

conn = sqlite3.connect("coffee_shop.db")
df.to_sql("orders", conn, if_exists="replace", index=False)
members_df.to_sql("members", conn, if_exists="replace", index=False)
events_df.to_sql("order_events", conn, if_exists="replace", index=False)
print("บันทึก coffee_orders.csv และ coffee_shop.db สำเร็จ")
display(df.head())


บันทึก coffee_orders.csv และ coffee_shop.db สำเร็จ


,order_id,queue_no,customer_name,member_id,menu_name,drink_type,size,topping,sweetness_level,receive_type,payment_method,subtotal,price,wait_minutes,status
0,1,Q001,ลีโอ สายทอง,None,อเมริกาโน่,เย็น,M,ไข่มุก,0%,ทานที่ร้าน,บัตร,65,65.0,15,เสร็จสิ้น
1,2,Q002,ลีโอ รักเรียน,M026,ลาเต้,ร้อน,L,ครีมชีส,100%,กลับบ้าน,บัตร,90,85.5,12,เสร็จสิ้น
2,3,Q003,นิโคล มั่นคง,M018,คาปูชิโน่,ปั่น,S,ไม่มี,50%,ทานที่ร้าน,บัตร,60,57.0,3,เสร็จสิ้น
3,4,Q004,ลูคัส มั่นคง,M047,มัทฉะลาเต้,เย็น,M,วิปครีม,0%,กลับบ้าน,เงินสด,80,76.0,2,เสร็จสิ้น
4,5,Q005,อเล็กซ์ รักเรียน,None,มัทฉะลาเต้,ปั่น,S,ไม่มี,75%,ทานที่ร้าน,บัตร,60,60.0,11,เสร็จสิ้น


# ขั้นที่ 7: วิเคราะห์ข้อมูลด้วย pandas และ SQL

ก่อนรัน ให้วาง `coffee_orders.csv` และ `coffee_shop.db` ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊ก หรืออัปโหลดเข้า Colab ก่อน


In [8]:
# @title
"""ขั้นที่ 7: วิเคราะห์ข้อมูลร้านกาแฟด้วย pandas และ SQL.""" # คำอธิบายภาพรวมของกระบวนการวิเคราะห์

from pathlib import Path # นำเข้าไลบรารีจัดการเส้นทางไฟล์
import sqlite3 # นำเข้าไลบรารีสำหรับจัดการฐานข้อมูล SQL

import pandas as pd # นำเข้าไลบรารี pandas สำหรับวิเคราะห์ข้อมูล


CSV_PATH = Path("coffee_orders.csv") # กำหนดตำแหน่งไฟล์ CSV
DATABASE_PATH = Path("coffee_shop.db") # กำหนดตำแหน่งไฟล์ฐานข้อมูล


def show(title, dataframe): # ฟังก์ชันสำหรับแสดงผลตารางข้อมูล
    """แสดง DataFrame ให้อ่านได้ง่าย.""" # คำอธิบายการทำงานของฟังก์ชัน
    print(f"\n{title}") # พิมพ์ชื่อหัวข้อการวิเคราะห์
    print(dataframe.to_string(index=False)) # พิมพ์ข้อมูลโดยไม่แสดงเลขลำดับแถว


if not CSV_PATH.exists(): # ตรวจสอบการมีอยู่ของไฟล์ CSV
    raise FileNotFoundError("ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน") # แจ้งข้อผิดพลาดหากไม่พบไฟล์
if not DATABASE_PATH.exists(): # ตรวจสอบการมีอยู่ของไฟล์ฐานข้อมูล
    raise FileNotFoundError("ไม่พบ coffee_shop.db กรุณารันขั้นที่ 5-6 ก่อน") # แจ้งข้อผิดพลาดหากไม่พบฐานข้อมูล


# 7.1 โหลด CSV และสำรวจข้อมูลเบื้องต้น # เริ่มขั้นตอนโหลดและสำรวจข้อมูล
coffee_df = pd.read_csv(CSV_PATH) # โหลดข้อมูลจาก CSV เข้าสู่ DataFrame
coffee_df.info() # แสดงรายละเอียดโครงสร้างข้อมูลเบื้องต้น
show("สถิติเบื้องต้น", coffee_df.describe(include="all").reset_index()) # แสดงค่าสถิติพื้นฐานของข้อมูลทั้งหมด

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         300 non-null    int64  
 1   queue_no         300 non-null    object 
 2   customer_name    300 non-null    object 
 3   member_id        125 non-null    object 
 4   menu_name        300 non-null    object 
 5   drink_type       300 non-null    object 
 6   size             300 non-null    object 
 7   topping          300 non-null    object 
 8   sweetness_level  300 non-null    object 
 9   receive_type     300 non-null    object 
 10  payment_method   300 non-null    object 
 11  subtotal         300 non-null    int64  
 12  price            300 non-null    float64
 13  wait_minutes     300 non-null    int64  
 14  status           300 non-null    object 
dtypes: float64(1), int64(3), object(11)
memory usage: 35.3+ KB

สถิติเบื้องต้น
 index   order_id queue_no custom

In [9]:
# 7.2 วิเคราะห์ด้วย pandas: groupby + agg + sort_values + Top 5 # ขั้นตอนการวิเคราะห์เชิงลึกด้วย pandas
menu_summary = ( # เริ่มต้นสร้างตัวแปรสรุปข้อมูลเมนู
    coffee_df.groupby("menu_name") # จัดกลุ่มข้อมูลตามชื่อเมนู
    .agg( # คำนวณค่าทางสถิติตามกลุ่ม
        total_orders=("order_id", "count"), # นับจำนวนคำสั่งซื้อรวม
        total_revenue=("price", "sum"), # คำนวณยอดขายรวม
        avg_price=("price", "mean"), # คำนวณราคาเฉลี่ยต่อหน่วย
        avg_wait_minutes=("wait_minutes", "mean"), # คำนวณเวลารอเฉลี่ย
    )
    .reset_index() # แปลงกลุ่มข้อมูลกลับเป็นคอลัมน์ปกติ
    .sort_values("total_revenue", ascending=False) # เรียงลำดับตามรายได้สูงสุดไปต่ำสุด
)
show("Top 5 เมนูที่สร้างรายได้สูงสุด", menu_summary.head(5)) # แสดงผลการวิเคราะห์ 5 อันดับแรก


Top 5 เมนูที่สร้างรายได้สูงสุด
 menu_name  total_orders  total_revenue  avg_price  avg_wait_minutes
     ลาเต้            29        2031.00  70.034483          9.344828
      นมสด            31        1972.50  63.629032          9.774194
    มอคค่า            27        1931.75  71.546296          8.962963
อเมริกาโน่            28        1917.75  68.491071         10.892857
 คาปูชิโน่            23        1643.75  71.467391         10.304348


In [10]:
# 7.3 วิเคราะห์ด้วย SQL # ขั้นตอนการวิเคราะห์ข้อมูลด้วย SQL Query
queries = { # สร้างชุดคำสั่ง SQL
    "ออเดอร์ราคาสูง": """ -- ค้นหาเมนูที่ราคาสูงกว่า 75 บาท
        SELECT order_id, queue_no, menu_name, price
        FROM orders
        WHERE price >= 75
        ORDER BY price DESC
        LIMIT 5
    """,
    "QR Code และกลับบ้าน": """ -- ค้นหาการสั่งกลับบ้านที่จ่ายด้วย QR Code
        SELECT order_id, menu_name, receive_type, payment_method, price
        FROM orders
        WHERE receive_type = 'กลับบ้าน'
          AND payment_method = 'QR Code'
        ORDER BY order_id
        LIMIT 5
    """,
    "คิวรอนาน": """ -- ค้นหาออเดอร์ที่ใช้เวลารอนานกว่า 15 นาที
        SELECT order_id, menu_name, wait_minutes
        FROM orders
        WHERE wait_minutes >= 15
        ORDER BY wait_minutes DESC, order_id
        LIMIT 5
    """,
    "รายได้ตามช่องทางชำระ": """ -- สรุปรายได้แยกตามวิธีการชำระเงิน
        SELECT payment_method,
               COUNT(*) AS total_orders,
               ROUND(SUM(price), 2) AS total_revenue,
               ROUND(AVG(price), 2) AS avg_price
        FROM orders
        GROUP BY payment_method
        ORDER BY total_revenue DESC
    """,
    "JOIN สมาชิกกับออเดอร์": """ -- เชื่อมข้อมูลการสั่งซื้อกับรายชื่อสมาชิก
        SELECT o.order_id, o.queue_no, m.member_name, o.menu_name, o.price
        FROM orders AS o
        JOIN members AS m ON o.member_id = m.member_id
        WHERE o.price >= 60
        ORDER BY o.price DESC
        LIMIT 5
    """,
} # สิ้นสุดการกำหนด Dictionary

with sqlite3.connect(DATABASE_PATH) as connection: # เปิดการเชื่อมต่อฐานข้อมูล SQLite
    for title, query in queries.items(): # วนลูปประมวลผลคำสั่ง SQL ทีละคำสั่ง
        show(title, pd.read_sql_query(query, connection)) # อ่านผลลัพธ์ SQL และแสดงผลทางหน้าจอ


ออเดอร์ราคาสูง
 order_id queue_no  menu_name  price
       41     Q041  ช็อกโกแลต  100.0
      198     Q198 เอสเพรสโซ่  100.0
       21     Q021 อเมริกาโน่   95.0
      168     Q168      ลาเต้   95.0
      186     Q186  ช็อกโกแลต   95.0

QR Code และกลับบ้าน
 order_id          menu_name receive_type payment_method  price
        9         มัทฉะลาเต้     กลับบ้าน        QR Code  70.00
       20          คาปูชิโน่     กลับบ้าน        QR Code  55.00
       22          คาปูชิโน่     กลับบ้าน        QR Code  85.00
       26 สตรอว์เบอร์รี่โซดา     กลับบ้าน        QR Code  76.00
       31               นมสด     กลับบ้าน        QR Code  52.25

คิวรอนาน
 order_id  menu_name  wait_minutes
       28   ยูซุโซดา            18
       67 เอสเพรสโซ่            18
       75 เอสเพรสโซ่            18
       95       นมสด            18
       98   ยูซุโซดา            18

รายได้ตามช่องทางชำระ
payment_method  total_orders  total_revenue  avg_price
          บัตร           106        7118.75      67.16
     

In [12]:
import pandas as pd

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "kunphat510214-netizen/Final-Final/main/"
)

menu_bq = pd.read_csv(
    BASE_URL + "summary_menu_performance.csv"
)
customer_bq = pd.read_csv(
    BASE_URL + "summary_customer_type.csv"
)
drink_bq = pd.read_csv(
    BASE_URL + "summary_drink_type.csv"
)

display(menu_bq)
display(customer_bq)
display(drink_bq)

,menu_name,total_orders,total_revenue,average_price,average_wait_minutes
0,ลาเต้,29,2031.00,70.03,9.34
1,นมสด,31,1972.50,63.63,9.77
2,มอคค่า,27,1931.75,71.55,8.96
3,อเมริกาโน่,28,1917.75,68.49,10.89
4,คาปูชิโน่,23,1643.75,71.47,10.30
5,มัทฉะลาเต้,23,1621.00,70.48,9.48
6,เอสเพรสโซ่,21,1457.25,69.39,12.62
7,ยูซุโซดา,24,1346.00,56.08,9.75
8,บลูฮาวายโซดา,22,1329.25,60.42,10.23
9,ช็อกโกแลต,17,1318.00,77.53,11.18


,customer_type,total_orders,total_revenue,average_spending,average_wait_minutes
0,Guest,175,12170.00,69.54,10.13
1,Member,125,7994.25,63.95,10.31


,drink_type,total_orders,total_revenue,average_price,average_wait_minutes
0,ร้อน,112,6972.50,62.25,10.71
1,ปั่น,94,6750.00,71.81,9.86
2,เย็น,94,6441.75,68.53,9.96


# ขั้นที่ 8: สร้างกราฟและสรุปผล

ก่อนรัน ให้วาง `coffee_orders.csv` ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊ก หรืออัปโหลดเข้า Colab ก่อน


In [ ]:
"""ขั้นที่ 8: สร้างกราฟและสรุปผลการวิเคราะห์ร้านกาแฟ."""

from pathlib import Path  # นำเข้าโมดูล Path สำหรับการจัดการพาธไฟล์

import matplotlib.pyplot as plt  # นำเข้า Matplotlib สำหรับการสร้างกราฟ
import pandas as pd  # นำเข้า Pandas สำหรับการจัดการข้อมูลในรูปแบบ DataFrame


CSV_PATH = Path("coffee_orders.csv")  # กำหนดพาธของไฟล์ CSV ที่มีข้อมูลออเดอร์กาแฟ
CHART_PATH = Path("coffee_analysis.png")  # กำหนดพาธสำหรับบันทึกไฟล์กราฟผลการวิเคราะห์

if not CSV_PATH.exists():  # ตรวจสอบว่าไฟล์ coffee_orders.csv มีอยู่หรือไม่
    raise FileNotFoundError("ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน")  # ถ้าไม่พบไฟล์ ให้แสดงข้อผิดพลาด


# โหลดข้อมูลใหม่เพื่อให้ไฟล์ขั้นที่ 8 รันแยกจากขั้นที่ 7 ได้
coffee_df = pd.read_csv(CSV_PATH)  # โหลดข้อมูลจากไฟล์ CSV เข้าสู่ DataFrame

menu_summary = (  # สร้าง DataFrame สรุปข้อมูลเมนู
    coffee_df.groupby("menu_name")  # จัดกลุ่มข้อมูลตามชื่อเมนู
    .agg(  # ใช้ฟังก์ชัน aggregate เพื่อคำนวณค่าต่างๆ
        total_orders=("order_id", "count"),  # นับจำนวนออเดอร์ทั้งหมดของแต่ละเมนู
        total_revenue=("price", "sum"),  # รวมรายได้ทั้งหมดของแต่ละเมนู
        avg_wait_minutes=("wait_minutes", "mean"),  # คำนวณเวลารอเฉลี่ยของแต่ละเมนู
    )
    .reset_index()  # รีเซ็ต index เพื่อให้ 'menu_name' กลับมาเป็นคอลัมน์ปกติ
    .sort_values("total_revenue", ascending=False)  # เรียงลำดับเมนูตามรายได้รวมจากมากไปน้อย
)
top5_menu = menu_summary.head(5)  # เลือก 5 อันดับแรกของเมนูที่มีรายได้สูงสุด
payment_counts = coffee_df["payment_method"].value_counts()  # นับจำนวนออเดอร์ตามวิธีการชำระเงิน

In [ ]:
# 8.1 สร้างกราฟอย่างน้อย 3 กราฟ
fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # สร้างพื้นที่สำหรับกราฟ 3 กราฟในแถวเดียวกัน

top5_menu.plot.bar(  # สร้างกราฟแท่งสำหรับ 5 อันดับเมนู
    x="menu_name",  # กำหนดแกน x เป็นชื่อเมนู
    y="total_revenue",  # กำหนดแกน y เป็นรายได้รวม
    ax=axes[0],  # วางในตำแหน่งกราฟที่ 1
    legend=False,  # ไม่แสดงคำอธิบายสัญลักษณ์
    color="steelblue",  # กำหนดสีแท่งกราฟ
)
axes[0].set_title("Top 5 Menu Revenue")  # ตั้งชื่อหัวข้อกราฟที่ 1
axes[0].set_xlabel("Menu")  # ตั้งชื่อแกน x
axes[0].set_ylabel("Revenue (THB)")  # ตั้งชื่อแกน y

payment_counts.plot.bar(ax=axes[1], color="seagreen")  # สร้างกราฟแท่งจำนวนออเดอร์ตามวิธีชำระเงินในตำแหน่งที่ 2
axes[1].set_title("Orders by Payment Method")  # ตั้งชื่อหัวข้อกราฟที่ 2
axes[1].set_xlabel("Payment Method")  # ตั้งชื่อแกน x
axes[1].set_ylabel("Orders")  # ตั้งชื่อแกน y

coffee_df["wait_minutes"].plot.hist(  # สร้างกราฟฮิสโตแกรมแสดงการกระจายของเวลารอ
    bins=9,  # แบ่งช่วงข้อมูลเป็น 9 ช่วง
    ax=axes[2],  # วางในตำแหน่งกราฟที่ 3
    color="darkorange",  # กำหนดสีส้ม
    edgecolor="black",  # กำหนดสีขอบแท่ง
)
axes[2].set_title("Queue Waiting Time")  # ตั้งชื่อหัวข้อกราฟที่ 3
axes[2].set_xlabel("Minutes")  # ตั้งชื่อแกน x
axes[2].set_ylabel("Orders")  # ตั้งชื่อแกน y

plt.tight_layout()  # ปรับการวางเลย์เอาต์ของกราฟให้เหมาะสมไม่ซ้อนทับกัน
plt.savefig(CHART_PATH, dpi=150, bbox_inches="tight")  # บันทึกรูปภาพกราฟลงไฟล์
plt.show()  # แสดงกราฟออกมาบนหน้าจอ

In [ ]:
# 8.2 สรุปผลอย่างน้อย 3-5 ประโยค
best_menu = menu_summary.iloc[0]  # ดึงข้อมูลเมนูที่อยู่อันดับแรก (รายได้สูงสุด)
member_share = coffee_df["member_id"].notna().mean() * 100  # คำนวณเปอร์เซ็นต์ของลูกค้าที่เป็นสมาชิก
popular_payment = payment_counts.index[0]  # ดึงชื่อวิธีการชำระเงินที่นิยมที่สุด
average_wait = coffee_df["wait_minutes"].mean()  # คำนวณค่าเฉลี่ยของเวลารอคิว

print(  # แสดงผลสรุปข้อที่ 1 เรื่องเมนูรายได้สูงสุด
    f"1. เมนูที่สร้างรายได้สูงสุดคือ {best_menu['menu_name']} "
    f"รวม {best_menu['total_revenue']:,.2f} บาท"
)
print(f"2. ลูกค้าสมาชิกคิดเป็น {member_share:.1f}% ของออเดอร์ทั้งหมด")  # แสดงผลสรุปข้อที่ 2 เรื่องสมาชิก
print(f"3. ช่องทางชำระเงินที่ใช้มากที่สุดคือ {popular_payment}")  # แสดงผลสรุปข้อที่ 3 เรื่องการชำระเงิน
print(f"4. เวลารอเฉลี่ยอยู่ที่ {average_wait:.1f} นาที")  # แสดงผลสรุปข้อที่ 4 เรื่องเวลารอ
print("5. ร้านควรเตรียมวัตถุดิบเมนูยอดนิยมและเพิ่มกำลังคนเมื่อคิวรอนาน")  # แสดงข้อเสนอแนะเพิ่มเติม
print(f"บันทึกกราฟไว้ที่ {CHART_PATH}")  # แจ้งตำแหน่งที่บันทึกไฟล์รูปภาพ